In [1]:
import pandas as pd
import numpy as np

In [2]:
riders = pd.read_csv("../dataset/processed/riders_features.csv")

print(riders.shape)

riders.head()

(45493, 33)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,Day_of_Week,Month,Weekend,Peak_Period,Trip_Distance_km,Traffic_Score,Weather_Score,Vehicle_Score,Rider_Experience,Workload
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,2022-02-12,21:55,...,Saturday,2,1,Dinner,10.280582,4,4,4,151.2,3.0
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,2022-02-13,14:55,...,Sunday,2,1,Lunch,6.242319,3,5,4,98.7,1.0
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,2022-03-04,17:30,...,Friday,3,0,Normal,13.787860,2,6,3,108.1,1.0
3,0xc8b6,LUDHRES15DEL02,34.0,4.3,30.899584,75.809346,30.919584,75.829346,2022-02-13,09:20,...,Sunday,2,1,Breakfast,2.930258,1,6,4,146.2,0.0
4,0xdb64,KNPRES14DEL02,24.0,4.7,26.463504,80.372929,26.593504,80.502929,2022-02-14,19:50,...,Monday,2,0,Dinner,19.396618,4,4,3,112.8,1.0


In [3]:
peak_map = {
    "Normal":0,
    "Breakfast":1,
    "Lunch":2,
    "Dinner":3
}

festival_map = {
    "No":0,
    "Yes":1
}

city_map = {
    city:i
    for i,city in enumerate(sorted(riders["City"].unique()))
}

order_map = {
    order:i
    for i,order in enumerate(sorted(riders["Type_of_order"].unique()))
}

O2A MOD

In [4]:
o2a = pd.DataFrame()

o2a["Traffic_Score"] = riders["Traffic_Score"]

o2a["Workload"] = riders["Workload"]

o2a["Multiple_Deliveries"] = riders["multiple_deliveries"]

o2a["Peak"] = riders["Peak_Period"].map(peak_map)

o2a["Festival"] = riders["Festival"].map(festival_map)

o2a["Rider_Experience"] = riders["Rider_Experience"]

o2a["Ratings"] = riders["Delivery_person_Ratings"]

In [5]:
o2a["Traffic_Workload"] = (
    o2a["Traffic_Score"] *
    o2a["Workload"]
)

o2a["Demand_Index"] = (
    o2a["Traffic_Score"] *
    (o2a["Peak"] + 1)
)

o2a["Rider_Load"] = (
    o2a["Rider_Experience"] /
    (o2a["Multiple_Deliveries"] + 1)
)

FM MOD

In [6]:
fm = pd.DataFrame()

fm["Trip_Distance"] = riders["Trip_Distance_km"]

fm["Traffic"] = riders["Traffic_Score"]

fm["Vehicle"] = riders["Vehicle_Score"]

fm["Vehicle_Condition"] = riders["Vehicle_condition"]

fm["Restaurant_Lat"] = riders["Restaurant_latitude"]

fm["Restaurant_Lon"] = riders["Restaurant_longitude"]

fm["Ratings"] = riders["Delivery_person_Ratings"]

In [7]:
fm["Travel_Index"] = (
    fm["Trip_Distance"] *
    fm["Traffic"]
)

fm["Vehicle_Index"] = (
    fm["Vehicle"] *
    fm["Vehicle_Condition"]
)

fm["Efficiency"] = (
    fm["Ratings"] *
    fm["Vehicle"]
)

WT MOD

In [8]:
wt = pd.DataFrame()

wt["Weather"] = riders["Weather_Score"]

wt["Peak"] = riders["Peak_Period"].map(peak_map)

wt["Festival"] = riders["Festival"].map(festival_map)

wt["City"] = riders["City"].map(city_map)

wt["Order"] = riders["Type_of_order"].map(order_map)

In [9]:
wt["Restaurant_Demand"] = (
    wt["Peak"] +
    wt["Festival"]
)

wt["Weather_Delay"] = (
    wt["Weather"] *
    (wt["Peak"] + 1)
)

LM MOD

In [10]:
lm = pd.DataFrame()

lm["Trip_Distance"] = riders["Trip_Distance_km"]

lm["Traffic"] = riders["Traffic_Score"]

lm["Weather"] = riders["Weather_Score"]

lm["Vehicle"] = riders["Vehicle_Score"]

lm["Lat"] = riders["Delivery_location_latitude"]

lm["Lon"] = riders["Delivery_location_longitude"]

lm["Experience"] = riders["Rider_Experience"]

In [11]:
lm["Delivery_Index"] = (
    lm["Trip_Distance"] *
    lm["Traffic"]
)

lm["Weather_Impact"] = (
    lm["Weather"] *
    lm["Trip_Distance"]
)

lm["Experience_Index"] = (
    lm["Experience"] /
    (lm["Traffic"] + 1)
)

In [12]:
fusion = pd.concat(

    [

        o2a,

        fm,

        wt,

        lm

    ],

    axis=1

)

In [13]:
fusion = fusion.loc[:,~fusion.columns.duplicated()]

print(fusion.shape)

(45493, 30)


In [14]:
global_features = riders[

    [

        "Order_Hour",

        "Pickup_Hour",

        "Weekend",

        "Month",

        "Delivery_person_Age"

    ]

]

fusion = pd.concat(

    [

        fusion,

        global_features

    ],

    axis=1

)

In [15]:
fusion["Time_taken (min)"] = riders["Time_taken (min)"]

In [16]:
fusion.to_csv(

    "../dataset/processed/adaptive_fusion_dataset.csv",

    index=False

)

print(fusion.shape)

fusion.head()

(45493, 36)


,Traffic_Score,Workload,Multiple_Deliveries,Peak,Festival,Rider_Experience,Ratings,Traffic_Workload,Demand_Index,Rider_Load,...,Experience,Delivery_Index,Weather_Impact,Experience_Index,Order_Hour,Pickup_Hour,Weekend,Month,Delivery_person_Age,Time_taken (min)
0,4,3.0,3.0,3,0,151.2,4.2,12.0,16,37.80,...,151.2,41.122328,41.122328,30.240000,21,22,1,2,36.0,46
1,3,1.0,1.0,2,0,98.7,4.7,3.0,9,49.35,...,98.7,18.726956,31.211593,24.675000,14,15,1,2,21.0,23
2,2,1.0,1.0,0,0,108.1,4.7,2.0,2,54.05,...,108.1,27.575720,82.727161,36.033333,17,17,0,3,23.0,21
3,1,0.0,0.0,1,0,146.2,4.3,0.0,2,146.20,...,146.2,2.930258,17.581547,73.100000,9,9,1,2,34.0,20
4,4,1.0,1.0,3,0,112.8,4.7,4.0,16,56.40,...,112.8,77.586473,77.586473,22.560000,19,20,0,2,24.0,41
